# 06 — Consolidated Validation + Notion Summary

Runs validation for ALL datasets, then generates comparison tables for Notion.

**Run AFTER all pipeline notebooks are complete.**

Pipelines available per dataset:
- **PBMC3k**: Scanpy, rsc, Seurat, ScaleSC
- **Lung65k**: Scanpy, rsc, Seurat, ScaleSC
- **Mouse Brain 1M**: Scanpy (1.15M), rsc (1M), ScaleSC (1M) — no Seurat

In [1]:
import subprocess, sys, json, os, glob
import pandas as pd
import numpy as np

CONFIG_PATH = "benchmark_config.json"
DATASETS = ["pbmc3k", "lung65k", "mouse_brain_1m"]

## Step 1: Run validation for each dataset

Uses the updated `validate_cross_pipeline_harmonized.py` (with mouse_brain_1m support).

In [ ]:
import time

for ds in DATASETS:
    print(f"\n{'='*72}")
    print(f"Running validation: {ds}")
    print(f"{'='*72}")
    t0 = time.time()
    result = subprocess.run(
        [sys.executable, "validate_cross_pipeline_harmonized.py",
         "--dataset", ds, "--config", CONFIG_PATH],
        capture_output=True, text=True,
    )
    elapsed = time.time() - t0
    if result.returncode == 0:
        print(result.stdout)
        print(f"Completed in {elapsed:.1f}s")
    else:
        print(f"FAILED (return code {result.returncode})")
        print("STDOUT:", result.stdout[-1000:] if result.stdout else "(empty)")
        print("STDERR:", result.stderr[-1000:] if result.stderr else "(empty)")


Running validation: pbmc3k
Validation — PBMC3k
Pipelines detected:
  scanpy_cpu   write/pbmc3k_scanpy_cpu_harmonized_clusters.csv
  rsc_gpu      write/pbmc3k_rsc_gpu_harmonized_clusters.csv
  seurat_cpu   write/pbmc3k_seurat_cpu_harmonized_clusters.csv
  scalesc_gpu  write/pbmc3k_scalesc_gpu_harmonized_clusters.csv

Summary:
dataset                comparison  common_cells  clusters_a  clusters_b      ARI      NMI  mean_dice  mean_native_deg_jaccard  mean_native_deg_spearman  mean_standardized_deg_jaccard  mean_standardized_deg_spearman  mean_module_profile_rho
 PBMC3k     scanpy_cpu vs rsc_gpu          2638           9           9 0.922560 0.934597   0.979719                 0.942537                  0.983240                       0.942537                        0.983240                 1.000000
 PBMC3k  scanpy_cpu vs seurat_cpu          2638           9          10 0.655449 0.761997   0.789819                 0.443738                  0.877465                       0.736301          

## Step 2: Load all results

In [ ]:
with open(CONFIG_PATH) as f:
    cfg = json.load(f)

all_summaries = []
all_details = {}

for ds in DATASETS:
    prefix = cfg["datasets"][ds]["pipeline_prefix"]
    val_dir = f"write/validation_{prefix}_harmonized"
    summary_csv = f"{val_dir}/{prefix}_validation_summary.csv"
    detail_json = f"{val_dir}/{prefix}_validation_details.json"
    
    if os.path.exists(summary_csv):
        df = pd.read_csv(summary_csv)
        all_summaries.append(df)
        print(f"Loaded {ds}: {len(df)} comparisons")
    else:
        print(f"MISSING: {summary_csv}")
    
    if os.path.exists(detail_json):
        with open(detail_json) as f:
            all_details[ds] = json.load(f)

if all_summaries:
    full_summary = pd.concat(all_summaries, ignore_index=True)
    print(f"\nTotal comparisons: {len(full_summary)}")

## Step 3: Notion-ready tables

### Table A — Core 3-pipeline comparison (Scanpy ↔ rsc ↔ Seurat/ScaleSC)

In [ ]:
# Filter to core comparisons only (exclude rsc version comparisons)
core_pairs = [
    "scanpy_cpu vs rsc_gpu",
    "scanpy_cpu vs seurat_cpu",
    "rsc_gpu vs seurat_cpu",
    "scanpy_cpu vs scalesc_gpu",
    "rsc_gpu vs scalesc_gpu",
    "seurat_cpu vs scalesc_gpu",
]

core = full_summary[full_summary["comparison"].isin(core_pairs)].copy()

# Select key columns
display_cols = [
    "dataset", "comparison", "common_cells",
    "clusters_a", "clusters_b", "ARI", "NMI", "mean_dice",
    "mean_standardized_deg_jaccard", "mean_standardized_deg_spearman",
    "mean_module_profile_rho",
]
core_display = core[[c for c in display_cols if c in core.columns]].copy()
core_display = core_display.round(3)

print("=" * 80)
print("TABLE A: Core pipeline comparisons")
print("=" * 80)
print(core_display.to_string(index=False))
core_display.to_csv("write/notion_table_a_core_comparisons.csv", index=False)
print("\nSaved: write/notion_table_a_core_comparisons.csv")

### Table B — Compact per-dataset summary (Notion-friendly)

In [ ]:
def make_compact_table(dataset_name, details_dict):
    """Create a compact comparison table for one dataset."""
    rows = []
    for pair_key, metrics in details_dict.items():
        # Parse pair names
        parts = pair_key.split("__vs__")
        if len(parts) != 2:
            continue
        name_a, name_b = parts
        
        # Skip rsc version comparisons for compact table
        if "0141" in name_a or "0141" in name_b or "015" in name_a or "015" in name_b:
            continue
        
        rows.append({
            "Comparison": f"{name_a} vs {name_b}",
            "Clusters": f"{metrics['n_clusters_a']} vs {metrics['n_clusters_b']}",
            "ARI": round(metrics["ARI"], 3),
            "NMI": round(metrics["NMI"], 3),
            "Mean Dice": round(metrics["mean_dice"], 3) if metrics.get("mean_dice") else "-",
            "Std DEG Jaccard": round(metrics["mean_standardized_deg_jaccard"], 3) if metrics.get("mean_standardized_deg_jaccard") else "-",
            "Native DEG Spearman": round(metrics["mean_native_deg_spearman"], 3) if metrics.get("mean_native_deg_spearman") else "-",
            "Module ρ": round(metrics["mean_module_profile_rho"], 3) if metrics.get("mean_module_profile_rho") else "-",
        })
    return pd.DataFrame(rows)


for ds in DATASETS:
    if ds not in all_details:
        print(f"\nSkipping {ds} (no details)")
        continue
    
    ds_name = cfg["datasets"][ds]["name"]
    print(f"\n{'='*72}")
    print(f"{ds_name}")
    print(f"{'='*72}")
    
    compact = make_compact_table(ds_name, all_details[ds])
    if len(compact) > 0:
        print(compact.to_string(index=False))
    else:
        # Show all pairs if filtering removed everything
        compact_all = []
        for pair_key, metrics in all_details[ds].items():
            parts = pair_key.split("__vs__")
            compact_all.append({
                "Comparison": f"{parts[0]} vs {parts[1]}",
                "ARI": round(metrics["ARI"], 3),
                "NMI": round(metrics["NMI"], 3),
                "Mean Dice": round(metrics.get("mean_dice", 0), 3),
                "Module ρ": round(metrics.get("mean_module_profile_rho", 0), 3),
            })
        print(pd.DataFrame(compact_all).to_string(index=False))

### Table C — Runtime comparison

In [ ]:
print("\n" + "=" * 72)
print("TABLE C: Runtime comparison")
print("=" * 72)

for ds in DATASETS:
    prefix = cfg["datasets"][ds]["pipeline_prefix"]
    ds_name = cfg["datasets"][ds]["name"]
    spec_files = sorted(glob.glob(f"write/{prefix}_*_spec.json"))
    
    if not spec_files:
        print(f"\n{ds_name}: No spec files found")
        continue
    
    print(f"\n--- {ds_name} ---")
    for sf in spec_files:
        with open(sf) as f:
            spec = json.load(f)
        pipeline = spec.get("pipeline", os.path.basename(sf))
        timings = spec.get("timings_seconds", {})
        total = sum(timings.values()) if timings else 0
        n_cells = spec.get("results", {}).get("n_cells", "?")
        n_clusters = spec.get("results", {}).get("n_clusters", "?")
        print(f"  {pipeline:30s} | {n_cells:>10} cells | {n_clusters:>3} clusters | {total:8.1f}s ({total/60:.1f} min)")

### Table D — Mouse Brain 1M: GPU Speedup

In [ ]:
# Compare Scanpy CPU vs rsc GPU step-by-step for mouse brain
prefix = cfg["datasets"]["mouse_brain_1m"]["pipeline_prefix"]

scanpy_spec = f"write/{prefix}_scanpy_cpu_harmonized_spec.json"
rsc_spec = f"write/{prefix}_rsc_gpu_harmonized_spec.json"

if os.path.exists(scanpy_spec) and os.path.exists(rsc_spec):
    with open(scanpy_spec) as f:
        s_spec = json.load(f)
    with open(rsc_spec) as f:
        r_spec = json.load(f)
    
    s_timings = s_spec.get("timings_seconds", {})
    r_timings = r_spec.get("timings_seconds", {})
    
    print("\n" + "=" * 72)
    print("TABLE D: Mouse Brain — Scanpy CPU vs rsc GPU step-by-step")
    print(f"Scanpy: {s_spec.get('results',{}).get('n_cells','?'):,} cells")
    print(f"rsc:    {r_spec.get('results',{}).get('n_cells','?'):,} cells")
    print("=" * 72)
    
    # Common steps
    steps = ["normalize_log1p", "hvg", "pca", "neighbors", "leiden", "umap", "de"]
    rows = []
    for step in steps:
        s_time = s_timings.get(step, None)
        r_time = r_timings.get(step, None)
        if s_time and r_time and r_time > 0:
            speedup = s_time / r_time
            rows.append({
                "Step": step,
                "Scanpy CPU (s)": round(s_time, 1),
                "rsc GPU (s)": round(r_time, 1),
                "Speedup": f"{speedup:.0f}×",
            })
    
    # Totals (excl I/O)
    s_total = sum(s_timings.get(s, 0) for s in steps)
    r_total = sum(r_timings.get(s, 0) for s in steps)
    if r_total > 0:
        rows.append({
            "Step": "TOTAL (excl I/O)",
            "Scanpy CPU (s)": round(s_total, 1),
            "rsc GPU (s)": round(r_total, 1),
            "Speedup": f"{s_total/r_total:.0f}×",
        })
    
    speedup_df = pd.DataFrame(rows)
    print(speedup_df.to_string(index=False))
    speedup_df.to_csv("write/notion_table_d_speedup.csv", index=False)
    print("\nSaved: write/notion_table_d_speedup.csv")
else:
    print("Missing spec files for mouse brain speedup comparison")

### Table E — Cross-dataset summary (key metrics only)

In [ ]:
print("\n" + "=" * 72)
print("TABLE E: Cross-dataset key metrics")
print("=" * 72)

summary_rows = []
for ds in DATASETS:
    if ds not in all_details:
        continue
    ds_name = cfg["datasets"][ds]["name"]
    for pair_key, metrics in all_details[ds].items():
        parts = pair_key.split("__vs__")
        if len(parts) != 2:
            continue
        comparison = f"{parts[0]} vs {parts[1]}"
        summary_rows.append({
            "Dataset": ds_name,
            "Comparison": comparison,
            "ARI": round(metrics["ARI"], 3),
            "NMI": round(metrics["NMI"], 3),
            "Module ρ": round(metrics.get("mean_module_profile_rho", 0), 3),
        })

cross_df = pd.DataFrame(summary_rows)
print(cross_df.to_string(index=False))
cross_df.to_csv("write/notion_table_e_cross_dataset.csv", index=False)
print("\nSaved: write/notion_table_e_cross_dataset.csv")

## Step 4: Copy-paste ready Notion markdown

In [ ]:
def df_to_notion_md(df, title=""):
    """Convert DataFrame to Notion-compatible markdown table."""
    lines = []
    if title:
        lines.append(f"### {title}\n")
    
    # Header
    cols = df.columns.tolist()
    lines.append("| " + " | ".join(str(c) for c in cols) + " |")
    lines.append("| " + " | ".join("---" for _ in cols) + " |")
    
    # Rows
    for _, row in df.iterrows():
        lines.append("| " + " | ".join(str(v) for v in row.values) + " |")
    
    return "\n".join(lines)


print("\n" + "=" * 72)
print("COPY-PASTE READY FOR NOTION")
print("=" * 72)

for ds in DATASETS:
    if ds not in all_details:
        continue
    ds_name = cfg["datasets"][ds]["name"]
    
    rows = []
    for pair_key, metrics in all_details[ds].items():
        parts = pair_key.split("__vs__")
        if len(parts) != 2:
            continue
        rows.append({
            "Comparison": f"{parts[0]} vs {parts[1]}",
            "Clusters": f"{metrics['n_clusters_a']} vs {metrics['n_clusters_b']}",
            "ARI": round(metrics["ARI"], 3),
            "NMI": round(metrics["NMI"], 3),
            "Dice": round(metrics.get("mean_dice", 0), 3),
            "DEG Jaccard": round(metrics.get("mean_standardized_deg_jaccard", 0), 3),
            "DEG ρ": round(metrics.get("mean_standardized_deg_spearman", 0), 3),
            "Module ρ": round(metrics.get("mean_module_profile_rho", 0), 3),
        })
    
    df = pd.DataFrame(rows)
    print(f"\n{df_to_notion_md(df, title=ds_name)}")
    print()

## Step 5: Save everything

In [ ]:
# Save full consolidated summary
if all_summaries:
    full_summary.to_csv("write/notion_full_summary_all_datasets.csv", index=False)
    print("Saved: write/notion_full_summary_all_datasets.csv")

print("\n=== All outputs ===")
for f in sorted(glob.glob("write/notion_*.csv")):
    print(f"  {f}")

print("\nDone! Copy the markdown tables above into Notion.")